In [23]:
import datasets
import os

In [24]:
# Path to the directory containing the HF datasets 
output_path = "/projects/iris/rferreir/hub_datasets"

In [12]:
dataset = datasets.load_dataset("ZhengyanShi/StepGame")

In [13]:
splits = ['train', 'test', 'validation']
for split in splits:
    dataset[split] = dataset[split].rename_column('label', 'answer').rename_column('story', 'scenario')

In [14]:
candidates = set()

for split in splits:
    ans =  dataset[split]['answer']
    for a in ans:
        candidates.add(a)

print(candidates)

{'lower-left', 'right', 'below', 'above', 'overlap', 'left', 'lower-right', 'upper-right', 'upper-left'}


In [15]:
candidates = list(candidates)

def add_candidates(row):
    row['candidates_answers'] = candidates
    return row

dataset = dataset.map(add_candidates)

In [16]:
print(dataset['train'][0])

{'scenario': ['X is to the left of K and is on the same horizontal plane.'], 'question': 'What is the relation of the agent X to the agent K?', 'answer': 'left', 'k_hop': 1, 'candidates_answers': ['lower-left', 'right', 'below', 'above', 'overlap', 'left', 'lower-right', 'upper-right', 'upper-left']}


In [17]:
def fuse_scenarios(row):
    row['scenario'] = ' '.join(row['scenario'])
    return row

dataset = dataset.map(fuse_scenarios)

Map: 100%|██████████| 100000/100000 [00:06<00:00, 15253.54 examples/s]


In [21]:
print(dataset['train'][49000])

{'scenario': 'If D is the center of a clock face, K is located between 2 and 3. K is at the lower side of X. X is diagonally to the bottom right of R. S and D are next to each other with S on the top and D at the bottom. M is placed at the bottom of R.', 'question': 'What is the relation of the agent K to the agent S?', 'answer': 'right', 'k_hop': 5, 'candidates_answers': ['lower-left', 'right', 'below', 'above', 'overlap', 'left', 'lower-right', 'upper-right', 'upper-left']}


In [30]:
for split in splits:
    print(dataset[split].features)

{'scenario': Value('string'), 'question': Value('string'), 'answer': Value('string'), 'k_hop': Value('int64'), 'candidates_answers': List(Value('string'))}
{'scenario': Value('string'), 'question': Value('string'), 'answer': Value('string'), 'k_hop': Value('int64'), 'candidates_answers': List(Value('string'))}
{'scenario': Value('string'), 'question': Value('string'), 'answer': Value('string'), 'k_hop': Value('int64'), 'candidates_answers': List(Value('string'))}


In [25]:
dataset.save_to_disk(os.path.join(output_path, f"StepGame"))

Saving the dataset (1/1 shards): 100%|██████████| 100000/100000 [00:00<00:00, 247439.74 examples/s]


In [31]:
import hashlib
import json

d2 = datasets.load_dataset("rfr2003/GeoBenchLLM", f"StepGame")
# We check that the content of the dataset is the same
def content_hash(ds):
    h = hashlib.sha256()
    for ex in ds:
        h.update(json.dumps(ex, sort_keys=True).encode())
    return h.hexdigest()

for split in splits:
    assert content_hash(dataset[split]) == content_hash(d2[split])

Generating test split: 100%|██████████| 100000/100000 [00:00<00:00, 218668.13 examples/s]
